Quick EDA of Olist_customer fields


In [15]:
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np


In [16]:


# Load the Olist customers dataset
data_path = Path("Olist E-Commerce Dataset") / "olist_customers_dataset.csv"
olist_df = pd.read_csv(data_path)

print(f"Loaded {len(olist_df):,} rows and {olist_df.shape[1]} columns from: {data_path}")
display(olist_df.head())

# Unique counts for requested fields
unique_city = olist_df["customer_city"].nunique(dropna=True)
unique_state = olist_df["customer_state"].nunique(dropna=True)

print(f"Unique customer_city values: {unique_city:,}")
print(f"Unique customer_state values: {unique_state:,}")

# Optional context: top values
print("\nTop 10 customer_city values by frequency:")
display(olist_df["customer_city"].value_counts(dropna=False).head(10))

print("\ncustomer_state distribution:")
display(olist_df["customer_state"].value_counts(dropna=False).sort_index())

Loaded 99,441 rows and 5 columns from: Olist E-Commerce Dataset\olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Unique customer_city values: 4,119
Unique customer_state values: 27

Top 10 customer_city values by frequency:


customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64


customer_state distribution:


customer_state
AC       81
AL      413
AM      148
AP       68
BA     3380
CE     1336
DF     2140
ES     2033
GO     2020
MA      747
MG    11635
MS      715
MT      907
PA      975
PB      536
PE     1652
PI      495
PR     5045
RJ    12852
RN      485
RO      253
RR       46
RS     5466
SC     3637
SE      350
SP    41746
TO      280
Name: count, dtype: int64

# Creating Source Codes

In [17]:
# Prepare ZIP prefix as a zero-padded 5-digit string
zip_prefix = (
    pd.to_numeric(olist_df["customer_zip_code_prefix"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

# Group by the first 4 digits
olist_df["zip_group_4"] = zip_prefix.str[:4]
group_count_4 = olist_df["zip_group_4"].nunique(dropna=True)

# Build source_code as STATE-first4zip (e.g., SP-1321)
state_code = olist_df["customer_state"].astype("string").str.strip().str.upper()
olist_df["source_code"] = state_code + "-" + olist_df["zip_group_4"]

print(f"Number of groups using first 4 digits: {group_count_4:,}")
print("Each group represents customers sharing the same first 4 digits of ZIP prefix.\n")

zip4_counts = (
    olist_df["zip_group_4"]
    .value_counts(dropna=False)
    .rename_axis("zip_group_4")
    .reset_index(name="customer_count")
)

print("Top 15 first-4-digit ZIP groups by customer count:")
display(zip4_counts.head(15))

print("Sample source_code values:")
display(olist_df[["customer_state", "customer_zip_code_prefix", "source_code"]].head(10))

Number of groups using first 4 digits: 5,699
Each group represents customers sharing the same first 4 digits of ZIP prefix.

Top 15 first-4-digit ZIP groups by customer count:


,zip_group_4,customer_count
0,1321,366
1,2279,334
2,1308,257
3,2910,253
4,3840,248
5,1224,242
6,1170,240
7,1305,239
8,1223,236
9,1304,215


Sample source_code values:


,customer_state,customer_zip_code_prefix,source_code
0,SP,14409,SP-1440
1,SP,9790,SP-0979
2,SP,1151,SP-0115
3,SP,8775,SP-0877
4,SP,13056,SP-1305
5,SC,89254,SC-8925
6,SP,4534,SP-0453
7,MG,35182,MG-3518
8,PR,81560,PR-8156
9,MG,30575,MG-3057


In [18]:
# Do the same grouping for first 3 and first 2 digits
for n in [3, 2]:
    col = f"zip_group_{n}"
    olist_df[col] = zip_prefix.str[:n]
    group_count = olist_df[col].nunique(dropna=True)
    
    print(f"Number of groups using first {n} digits: {group_count:,}")
    print(f"Each group represents customers sharing the same first {n} digits of ZIP prefix.\n")

# Side-by-side comparison of grouping granularity
summary = pd.DataFrame(
    {
        "prefix_digits": [4, 3, 2],
        "number_of_groups": [
            olist_df["zip_group_4"].nunique(dropna=True),
            olist_df["zip_group_3"].nunique(dropna=True),
            olist_df["zip_group_2"].nunique(dropna=True),
        ],
    }
).sort_values("prefix_digits", ascending=False)

print("Comparison of ZIP grouping depth:")
display(summary)

Number of groups using first 3 digits: 851
Each group represents customers sharing the same first 3 digits of ZIP prefix.

Number of groups using first 2 digits: 98
Each group represents customers sharing the same first 2 digits of ZIP prefix.

Comparison of ZIP grouping depth:


,prefix_digits,number_of_groups
0,4,5699
1,3,851
2,2,98


Grouping Olist customer by State-ZIP

In [19]:
count = olist_df["source_code"].value_counts(dropna=False)
proportions = olist_df["source_code"].value_counts(normalize=True, dropna=False)
print("Top 10 source_code proportions:")

source_code_summary = pd.DataFrame({
    "count": count,
    "proportion": proportions,
    "percentage": proportions * 100
})

source_code_summary.head(20)

Top 10 source_code proportions:


,count,proportion,percentage
source_code,,,
SP-1321,366,0.003681,0.368057
RJ-2279,334,0.003359,0.335878
SP-1308,257,0.002584,0.258445
ES-2910,253,0.002544,0.254422
MG-3840,248,0.002494,0.249394
SP-1224,242,0.002434,0.24336
SP-1170,240,0.002413,0.241349
SP-1305,239,0.002403,0.240344
SP-1223,236,0.002373,0.237327


In [20]:
# Count how many rows belong to each source_code
olist_df["source_code_count"] = (
    olist_df.groupby("source_code", dropna=False)["source_code"]
      .transform("size")
)

# Proportion of all rows that have that row's source_code
olist_df["source_code_proportion"] = olist_df["source_code_count"] / len(olist_df)

# Percentage of all rows that have that row's source_code
olist_df["source_code_percentage"] = olist_df["source_code_proportion"] * 100

## Source Code Validation Checks

In [21]:
# ============================================
# Olist Source Code Validation - Basic Checks
# ============================================

required_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "zip_group_4",
    "zip_group_3",
    "zip_group_2",
    "source_code",
    "source_code_count",
    "source_code_proportion",
    "source_code_percentage"
]

print("==== BASIC DATAFRAME CHECKS ====")
print(f"Rows: {len(olist_df):,}")
print(f"Columns: {olist_df.shape[1]:,}")

missing_required_cols = [col for col in required_columns if col not in olist_df.columns]

if missing_required_cols:
    print("Missing required columns:")
    print(missing_required_cols)
else:
    print("All required columns exist.")

print("\n==== MISSING VALUE COUNTS ====")
missing_summary = (
    olist_df[required_columns]
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(olist_df) * 100
).round(4)

display(missing_summary)

==== BASIC DATAFRAME CHECKS ====
Rows: 99,441
Columns: 12
All required columns exist.

==== MISSING VALUE COUNTS ====


,column,missing_count,missing_percentage
0,customer_id,0,0.0
1,customer_unique_id,0,0.0
2,customer_zip_code_prefix,0,0.0
3,customer_city,0,0.0
4,customer_state,0,0.0
5,zip_group_4,0,0.0
6,zip_group_3,0,0.0
7,zip_group_2,0,0.0
8,source_code,0,0.0
9,source_code_count,0,0.0


In [23]:
# ============================================
# Customer ID Validation
# ============================================

print("==== CUSTOMER ID CHECKS ====")

customer_id_nulls = olist_df["customer_id"].isna().sum()
customer_unique_id_nulls = olist_df["customer_unique_id"].isna().sum()

customer_id_duplicates = olist_df["customer_id"].duplicated().sum()
customer_unique_id_duplicates = olist_df["customer_unique_id"].duplicated().sum()

print(f"Missing customer_id values: {customer_id_nulls:,}")
print(f"Missing customer_unique_id values: {customer_unique_id_nulls:,}")
print(f"Duplicate customer_id values: {customer_id_duplicates:,}")
print(f"Duplicate customer_unique_id values: {customer_unique_id_duplicates:,}")

if customer_id_duplicates > 0:
    print("\nDuplicate customer_id examples:")
    display(
        olist_df[olist_df["customer_id"].duplicated(keep=False)]
        .sort_values("customer_id")
        .head(20)
    )
else:
    print("customer_id appears unique.")

==== CUSTOMER ID CHECKS ====
Missing customer_id values: 0
Missing customer_unique_id values: 0
Duplicate customer_id values: 0
Duplicate customer_unique_id values: 3,345
customer_id appears unique.


In [24]:
# ============================================
# ZIP Prefix Validation
# ============================================

print("==== ZIP PREFIX CHECKS ====")

zip_as_string = (
    pd.to_numeric(olist_df["customer_zip_code_prefix"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

olist_df["_validated_zip_prefix"] = zip_as_string

invalid_zip_numeric = pd.to_numeric(
    olist_df["customer_zip_code_prefix"],
    errors="coerce"
).isna().sum()

invalid_zip_length = (
    olist_df["_validated_zip_prefix"].str.len() != 5
).sum()

invalid_zip_digits = (
    ~olist_df["_validated_zip_prefix"].str.match(r"^\d{5}$", na=False)
).sum()

print(f"Non-numeric ZIP prefix values: {invalid_zip_numeric:,}")
print(f"ZIP prefixes not length 5 after padding: {invalid_zip_length:,}")
print(f"ZIP prefixes not matching 5 digits: {invalid_zip_digits:,}")

bad_zip_rows = olist_df[
    pd.to_numeric(olist_df["customer_zip_code_prefix"], errors="coerce").isna()
    | (olist_df["_validated_zip_prefix"].str.len() != 5)
    | (~olist_df["_validated_zip_prefix"].str.match(r"^\d{5}$", na=False))
]

if len(bad_zip_rows) > 0:
    print("\nProblematic ZIP prefix rows:")
    display(bad_zip_rows.head(20))
else:
    print("All ZIP prefixes are valid 5-digit prefixes after zero-padding.")

==== ZIP PREFIX CHECKS ====
Non-numeric ZIP prefix values: 0
ZIP prefixes not length 5 after padding: 0
ZIP prefixes not matching 5 digits: 0
All ZIP prefixes are valid 5-digit prefixes after zero-padding.


In [25]:
# ============================================
# ZIP Group Validation
# ============================================

print("==== ZIP GROUP CHECKS ====")

expected_zip_group_4 = olist_df["_validated_zip_prefix"].str[:4]
expected_zip_group_3 = olist_df["_validated_zip_prefix"].str[:3]
expected_zip_group_2 = olist_df["_validated_zip_prefix"].str[:2]

zip_group_4_mismatch = (
    olist_df["zip_group_4"].astype("string") != expected_zip_group_4
)

zip_group_3_mismatch = (
    olist_df["zip_group_3"].astype("string") != expected_zip_group_3
)

zip_group_2_mismatch = (
    olist_df["zip_group_2"].astype("string") != expected_zip_group_2
)

print(f"zip_group_4 mismatches: {zip_group_4_mismatch.sum():,}")
print(f"zip_group_3 mismatches: {zip_group_3_mismatch.sum():,}")
print(f"zip_group_2 mismatches: {zip_group_2_mismatch.sum():,}")

if zip_group_4_mismatch.sum() > 0:
    print("\nzip_group_4 mismatch examples:")
    display(
        olist_df.loc[
            zip_group_4_mismatch,
            ["customer_zip_code_prefix", "_validated_zip_prefix", "zip_group_4"]
        ].head(20)
    )

if zip_group_3_mismatch.sum() > 0:
    print("\nzip_group_3 mismatch examples:")
    display(
        olist_df.loc[
            zip_group_3_mismatch,
            ["customer_zip_code_prefix", "_validated_zip_prefix", "zip_group_3"]
        ].head(20)
    )

if zip_group_2_mismatch.sum() > 0:
    print("\nzip_group_2 mismatch examples:")
    display(
        olist_df.loc[
            zip_group_2_mismatch,
            ["customer_zip_code_prefix", "_validated_zip_prefix", "zip_group_2"]
        ].head(20)
    )

==== ZIP GROUP CHECKS ====
zip_group_4 mismatches: 0
zip_group_3 mismatches: 0
zip_group_2 mismatches: 0


In [26]:
# ============================================
# Customer State Validation
# ============================================

print("==== CUSTOMER STATE CHECKS ====")

valid_brazil_states = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
}

state_clean = (
    olist_df["customer_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

state_nulls = olist_df["customer_state"].isna().sum()
state_length_invalid = (state_clean.str.len() != 2).sum()
state_not_uppercase = (
    olist_df["customer_state"].astype("string") != state_clean
).sum()
state_invalid_code = (~state_clean.isin(valid_brazil_states)).sum()

print(f"Missing customer_state values: {state_nulls:,}")
print(f"State codes not length 2: {state_length_invalid:,}")
print(f"State codes not already clean uppercase: {state_not_uppercase:,}")
print(f"Invalid Brazilian state codes: {state_invalid_code:,}")

invalid_state_rows = olist_df[
    state_clean.isna()
    | (state_clean.str.len() != 2)
    | (~state_clean.isin(valid_brazil_states))
]

if len(invalid_state_rows) > 0:
    print("\nInvalid state rows:")
    display(
        invalid_state_rows[
            ["customer_id", "customer_unique_id", "customer_state", "customer_city"]
        ].head(20)
    )
else:
    print("All customer_state values look valid.")

==== CUSTOMER STATE CHECKS ====
Missing customer_state values: 0
State codes not length 2: 0
State codes not already clean uppercase: 0
Invalid Brazilian state codes: 0
All customer_state values look valid.


In [27]:
# ============================================
# Source Code Validation
# ============================================

print("==== SOURCE CODE CHECKS ====")

expected_source_code = (
    state_clean
    + "-"
    + expected_zip_group_4
)

source_code_mismatch = (
    olist_df["source_code"].astype("string") != expected_source_code
)

print(f"source_code mismatches: {source_code_mismatch.sum():,}")
print(f"Unique source_code values: {olist_df['source_code'].nunique(dropna=True):,}")

if source_code_mismatch.sum() > 0:
    print("\nsource_code mismatch examples:")
    display(
        olist_df.loc[
            source_code_mismatch,
            [
                "customer_state",
                "customer_zip_code_prefix",
                "zip_group_4",
                "source_code"
            ]
        ]
        .assign(expected_source_code=expected_source_code[source_code_mismatch])
        .head(20)
    )
else:
    print("All source_code values match expected STATE-zip4 format.")

==== SOURCE CODE CHECKS ====
source_code mismatches: 0
Unique source_code values: 5,699
All source_code values match expected STATE-zip4 format.


In [28]:
# ============================================
# Source Code Weight Validation
# ============================================

print("==== SOURCE CODE WEIGHT CHECKS ====")

expected_source_code_count = (
    olist_df
    .groupby("source_code", dropna=False)["source_code"]
    .transform("size")
)

expected_source_code_proportion = expected_source_code_count / len(olist_df)
expected_source_code_percentage = expected_source_code_proportion * 100

count_mismatch = (
    olist_df["source_code_count"] != expected_source_code_count
)

proportion_mismatch = ~np.isclose(
    olist_df["source_code_proportion"],
    expected_source_code_proportion,
    rtol=0,
    atol=1e-12
)

percentage_mismatch = ~np.isclose(
    olist_df["source_code_percentage"],
    expected_source_code_percentage,
    rtol=0,
    atol=1e-10
)

print(f"source_code_count mismatches: {count_mismatch.sum():,}")
print(f"source_code_proportion mismatches: {proportion_mismatch.sum():,}")
print(f"source_code_percentage mismatches: {percentage_mismatch.sum():,}")

if count_mismatch.sum() > 0:
    print("\nsource_code_count mismatch examples:")
    display(
        olist_df.loc[
            count_mismatch,
            ["source_code", "source_code_count"]
        ]
        .assign(expected_count=expected_source_code_count[count_mismatch])
        .head(20)
    )

if proportion_mismatch.sum() > 0:
    print("\nsource_code_proportion mismatch examples:")
    display(
        olist_df.loc[
            proportion_mismatch,
            ["source_code", "source_code_proportion"]
        ]
        .assign(expected_proportion=expected_source_code_proportion[proportion_mismatch])
        .head(20)
    )

if percentage_mismatch.sum() > 0:
    print("\nsource_code_percentage mismatch examples:")
    display(
        olist_df.loc[
            percentage_mismatch,
            ["source_code", "source_code_percentage"]
        ]
        .assign(expected_percentage=expected_source_code_percentage[percentage_mismatch])
        .head(20)
    )

==== SOURCE CODE WEIGHT CHECKS ====
source_code_count mismatches: 0
source_code_proportion mismatches: 0
source_code_percentage mismatches: 0


In [29]:
# ============================================
# Distinct Source Code Summary Validation
# ============================================

print("==== DISTINCT SOURCE CODE SUMMARY ====")

source_code_validation_summary = (
    olist_df
    .groupby("source_code", dropna=False)
    .agg(
        customer_rows=("customer_id", "count"),
        unique_customers=("customer_unique_id", "nunique"),
        row_level_count=("source_code_count", "first"),
        row_level_proportion=("source_code_proportion", "first"),
        row_level_percentage=("source_code_percentage", "first")
    )
    .reset_index()
)

source_code_validation_summary["expected_proportion"] = (
    source_code_validation_summary["customer_rows"] / len(olist_df)
)

source_code_validation_summary["expected_percentage"] = (
    source_code_validation_summary["expected_proportion"] * 100
)

source_code_validation_summary["count_is_correct"] = (
    source_code_validation_summary["customer_rows"]
    == source_code_validation_summary["row_level_count"]
)

source_code_validation_summary["proportion_is_correct"] = np.isclose(
    source_code_validation_summary["row_level_proportion"],
    source_code_validation_summary["expected_proportion"],
    rtol=0,
    atol=1e-12
)

source_code_validation_summary["percentage_is_correct"] = np.isclose(
    source_code_validation_summary["row_level_percentage"],
    source_code_validation_summary["expected_percentage"],
    rtol=0,
    atol=1e-10
)

print(f"Unique source_codes: {len(source_code_validation_summary):,}")
print(f"Total customer rows represented: {source_code_validation_summary['customer_rows'].sum():,}")
print(f"Sum of distinct source_code proportions: {source_code_validation_summary['expected_proportion'].sum():.12f}")
print(f"Sum of distinct source_code percentages: {source_code_validation_summary['expected_percentage'].sum():.8f}%")

bad_source_summary = source_code_validation_summary[
    ~source_code_validation_summary["count_is_correct"]
    | ~source_code_validation_summary["proportion_is_correct"]
    | ~source_code_validation_summary["percentage_is_correct"]
]

if len(bad_source_summary) > 0:
    print("\nBad source_code summary rows:")
    display(bad_source_summary.head(20))
else:
    print("All source_code counts, proportions, and percentages are correct.")

display(
    source_code_validation_summary
    .sort_values("customer_rows", ascending=False)
    .head(20)
)

==== DISTINCT SOURCE CODE SUMMARY ====
Unique source_codes: 5,699
Total customer rows represented: 99,441
Sum of distinct source_code proportions: 1.000000000000
Sum of distinct source_code percentages: 100.00000000%
All source_code counts, proportions, and percentages are correct.


,source_code,customer_rows,unique_customers,row_level_count,row_level_proportion,row_level_percentage,expected_proportion,expected_percentage,count_is_correct,proportion_is_correct,percentage_is_correct
5154,SP-1321,366,353,366,0.003681,0.368057,0.003681,0.368057,True,True,True
3185,RJ-2279,334,325,334,0.003359,0.335878,0.003359,0.335878,True,True,True
5144,SP-1308,257,247,257,0.002584,0.258445,0.002584,0.258445,True,True,True
887,ES-2910,253,246,253,0.002544,0.254422,0.002544,0.254422,True,True,True
1786,MG-3840,248,243,248,0.002494,0.249394,0.002494,0.249394,True,True,True
5085,SP-1224,242,231,242,0.002434,0.243360,0.002434,0.243360,True,True,True
5049,SP-1170,240,224,240,0.002413,0.241349,0.002413,0.241349,True,True,True
5141,SP-1305,239,235,239,0.002403,0.240344,0.002403,0.240344,True,True,True
5084,SP-1223,236,226,236,0.002373,0.237327,0.002373,0.237327,True,True,True
5140,SP-1304,215,210,215,0.002162,0.216209,0.002162,0.216209,True,True,True


In [30]:
# ============================================
# Source Code Distribution Sanity Checks
# ============================================

print("==== SOURCE CODE DISTRIBUTION SANITY CHECKS ====")

source_counts = (
    olist_df["source_code"]
    .value_counts(dropna=False)
    .reset_index()
)

source_counts.columns = ["source_code", "customer_count"]

print("Customer count per source_code:")
display(source_counts["customer_count"].describe())

print("\nTop 20 source_codes:")
display(source_counts.head(20))

print("\nBottom 20 source_codes:")
display(source_counts.tail(20))

total_rows = len(olist_df)

top_n_summary = []

for n in [10, 25, 50, 100, 250, 500, 1000, 2000]:
    top_n_customers = source_counts.head(n)["customer_count"].sum()
    top_n_summary.append({
        "top_n_source_codes": n,
        "customers_in_top_n": top_n_customers,
        "percentage_of_dataset": top_n_customers / total_rows * 100
    })

top_n_summary = pd.DataFrame(top_n_summary)

display(top_n_summary)

==== SOURCE CODE DISTRIBUTION SANITY CHECKS ====
Customer count per source_code:


count       5699.0
mean     17.448851
std      24.639428
min            1.0
25%            3.0
50%            9.0
75%           22.0
max          366.0
Name: customer_count, dtype: Float64


Top 20 source_codes:


,source_code,customer_count
0,SP-1321,366
1,RJ-2279,334
2,SP-1308,257
3,ES-2910,253
4,MG-3840,248
5,SP-1224,242
6,SP-1170,240
7,SP-1305,239
8,SP-1223,236
9,SP-1304,215



Bottom 20 source_codes:


,source_code,customer_count
5679,SP-1277,1
5680,SC-8943,1
5681,ES-2961,1
5682,PE-5531,1
5683,BA-4180,1
5684,SP-1941,1
5685,SE-4975,1
5686,RS-9639,1
5687,RN-5999,1
5688,PB-5855,1


,top_n_source_codes,customers_in_top_n,percentage_of_dataset
0,10,2630,2.644784
1,25,5226,5.255378
2,50,8503,8.550799
3,100,13835,13.912772
4,250,25338,25.480436
5,500,38976,39.195101
6,1000,57907,58.232520
7,2000,79566,80.013274


In [31]:
source_code_summary = source_code_summary.sort_values("count", ascending=False)
source_code_summary.head(20)

,count,proportion,percentage
source_code,,,
SP-1321,366,0.003681,0.368057
RJ-2279,334,0.003359,0.335878
SP-1308,257,0.002584,0.258445
ES-2910,253,0.002544,0.254422
MG-3840,248,0.002494,0.249394
SP-1224,242,0.002434,0.24336
SP-1170,240,0.002413,0.241349
SP-1305,239,0.002403,0.240344
SP-1223,236,0.002373,0.237327


In [33]:
source_code_summary.to_csv('source_code_summary.csv', index=True)